# 11 · Tokenizer 三家对比：GPT-2 / cl100k / o200k

> **学习目标**：从「为什么中文用 GPT API 这么费 token」的实际痛点出发，对比 OpenAI 三代 BPE 分词器的设计变化，并理解 BPE 的本质。
>
> **预备**：05 已过（懂 token / vocab 的概念）。
>
> **为什么重要**：分词器是「LLM 看世界的眼镜」。一个 query 是 100 token 还是 50 token，**直接 ×2 你的 API 费用**，还影响上下文窗口、首 token 延迟、模型对中文/代码的理解力。

**本机用 `tiktoken`（OpenAI 官方，offline）跑**。Qwen / Llama3 的 tokenizer 需要从 HF 下 `tokenizer.json`，本机网络不稳定，章节末给出网络通时的代码。

In [ ]:
import tiktoken
import matplotlib.pyplot as plt

# OpenAI 三代主流编码（按时间从早到新）
ENCODINGS = {
    'gpt2':         tiktoken.get_encoding('gpt2'),          # GPT-2 (2019)，vocab 50k
    'cl100k_base':  tiktoken.get_encoding('cl100k_base'),    # GPT-3.5 / GPT-4 (2022)，vocab 100k
    'o200k_base':   tiktoken.get_encoding('o200k_base'),     # GPT-4o (2024)，vocab 200k
}
for name, enc in ENCODINGS.items():
    print(f'{name:14}  vocab = {enc.n_vocab:>7,}')

## 1. 同一段文字，三家分词数量

测 4 类常见输入。**结论一句话**：vocab 越大 → 中文/代码越省，英文差距不大。

In [ ]:
SAMPLES = {
    '英文短句': 'The Transformer is an attention-based architecture introduced in 2017.',
    '中文短句': 'Transformer 是 2017 年提出的基于注意力的架构。',
    '中文长段': '检索增强生成（Retrieval-Augmented Generation, RAG）通过把外部知识库的检索结果拼进上下文，缓解了大模型幻觉问题，是当下企业落地最广泛的方案之一。',
    '代码片段': 'def softmax(x, axis=-1):\n    e = (x - x.max(axis=axis, keepdims=True)).exp()\n    return e / e.sum(axis=axis, keepdims=True)',
}

# 横向表
print(f'{"类型":<8} {"字符数":>6} | ' + ' | '.join(f'{name:>14}' for name in ENCODINGS))
print('-' * 70)
results = {name: [] for name in ENCODINGS}
labels = []
char_counts = []
for label, text in SAMPLES.items():
    labels.append(label)
    char_counts.append(len(text))
    counts = {name: len(enc.encode(text)) for name, enc in ENCODINGS.items()}
    for name, n in counts.items():
        results[name].append(n)
    cells = ' | '.join(f'{counts[name]:>9} tok' for name in ENCODINGS)
    print(f'{label:<8} {len(text):>6} | {cells}')

In [ ]:
# Bar chart 直观看
import numpy as np
x = np.arange(len(labels))
w = 0.25
plt.figure(figsize=(10, 4))
for i, (name, vals) in enumerate(results.items()):
    plt.bar(x + (i - 1) * w, vals, w, label=name)
plt.xticks(x, labels, rotation=0)
plt.ylabel('token 数')
plt.title('同一文本，3 个 tokenizer 的 token 数对比')
plt.legend(); plt.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

# 字符 / token 比（越高越省）
print('\n字符/token 比（越高越省）:')
for name in ENCODINGS:
    ratios = [c / t for c, t in zip(char_counts, results[name])]
    print(f'  {name:14}', '  '.join(f'{r:.2f}' for r in ratios))

## 2. 看 BPE 实际「切」成什么样

**BPE = Byte-Pair Encoding**：先按字节拆，再迭代合并「最常一起出现的字节对」直到 vocab 满。结果：常见词整块、罕见词拆碎、生僻中文一个字可能拆成 3 个字节。

In [ ]:
def show_pieces(text: str, max_show: int = 30):
    print(f'输入: {text!r}')
    for name, enc in ENCODINGS.items():
        ids = enc.encode(text)
        pieces = [enc.decode([i]) for i in ids[:max_show]]
        tail = '' if len(ids) <= max_show else f' ... (+{len(ids)-max_show} more)'
        print(f'  {name:14}  {len(ids):2d} tok: {pieces}{tail}')

show_pieces('Hello, world!')
print()
show_pieces('你好世界')
print()
show_pieces('注意力机制')

In [ ]:
# 极端案例：罕见汉字、emoji、数字
for text in ['饕餮', 'AI 2026', '🤖🚀', '1234567890']:
    show_pieces(text)
    print()

## 3. 一个常被忽视的设计：「数字单独」与「空格归属」

**数字单独 token 化**（o200k 引入）：让模型做数学不被「123 vs 1234 拆法不同」带偏。

**空格归属**：GPT-2 把 `' the'`（带前导空格）当一个 token，所以模型其实见的是「位置 + 词」而非纯词。

In [ ]:
# 数字的不同拆法
print('数字拆法（重点看 o200k）:')
for text in ['123', '1234', '12345', '123456']:
    show_pieces(text)
    print()

In [ ]:
# 空格归属：前导空格成为 token 的一部分
print('空格归属（重点看 piece 里的前导空格）:')
show_pieces('hello world the end')
print('\n→ 注意到 " hello"、" world"、" the"、" end" 都是「带前导空格的 token」')
print('→ 这就是为什么写 prompt 时 "hello" vs " hello" 模型行为可能不同。')

## 4. 中文负担量化 —— 一段 100 字中文实际花多少 USD

**假设按 GPT-4 cl100k 计费 $0.01/1k input tok**：

In [ ]:
long_zh = '人工智能的发展正在改变各行各业的工作方式，从医疗到教育，从金融到制造业，每一个领域都在探索如何利用大模型提升效率与体验。' * 5
long_en = 'The development of artificial intelligence is transforming every industry, from healthcare to education to finance to manufacturing, each sector exploring how to leverage large models to improve efficiency and experience. ' * 5

price_per_1k = 0.01    # 假设的 input 价（美元）
for label, text in [('中文 ~500 字', long_zh), ('英文 ~500 字 对照', long_en)]:
    n_chars = len(text)
    print(f'\n{label}（{n_chars} 字符 / 词）:')
    for name, enc in ENCODINGS.items():
        n_tok = len(enc.encode(text))
        usd = n_tok * price_per_1k / 1000
        print(f'  {name:14}  {n_tok:>5} tok   ${usd:.5f}')
print('\n→ 同样信息量，中文用 GPT-2 编码会贵 2~3 倍。o200k 把中文成本拉回与英文同量级。')
print('→ 这是为什么 GPT-4o 中文体验「感觉」比 GPT-4 流畅 —— 它能在同样窗口里塞下更多中文。')

## 5. 网络通时：加载 Qwen / Llama3 tokenizer

下面代码本机网络不稳定时会失败 —— **只在网络可用时跑**。它的作用是把对比从「3 家 OpenAI」扩到「OpenAI vs 国产 vs Llama」。

**预期结论**：Qwen 系列的中文 token 数会**显著低于** o200k_base（Qwen 是专为中文优化的）。

In [ ]:
# 仅在网络可用时跑
try:
    from tokenizers import Tokenizer
    import urllib.request

    # 直接 download tokenizer.json（比 transformers 轻量得多）
    QWEN_URL = 'https://huggingface.co/Qwen/Qwen2.5-0.5B/resolve/main/tokenizer.json'
    LOCAL_PATH = '_qwen_tokenizer.json'

    import os
    if not os.path.exists(LOCAL_PATH):
        print('downloading...')
        urllib.request.urlretrieve(QWEN_URL, LOCAL_PATH)

    qwen_tok = Tokenizer.from_file(LOCAL_PATH)
    print('Qwen 2.5 vocab:', qwen_tok.get_vocab_size())

    for label, text in SAMPLES.items():
        ids = qwen_tok.encode(text).ids
        print(f'  {label:<8}  {len(ids):>4} tok  (Qwen)')
except Exception as e:
    print(f'⚠ 无法加载 Qwen tokenizer（多半是网络问题）：{type(e).__name__}: {str(e)[:100]}')
    print('  跳过本节，OpenAI 三家对比已经够看主要规律。')

## 深入思考

1. **为什么 OpenAI 的 cl100k 不一开始就用 o200k？**
   - vocab 越大 embedding 表越大、输出 head 越大；推理时 softmax over vocab 也更贵。**是个工程权衡**，不是越大越好。
2. **BPE 算法核心一句话**：先字节分，再迭代地把「在训练语料里最常出现的相邻 token 对」合并成新 token，直到 vocab 满。
3. **Qwen / Llama3 / GPT-4o 都在改 tokenizer，意味着什么？**
   - tokenizer 是预训练**最早期就要定**的东西，改 tokenizer = 从头预训练。改一次成本巨大，但收益（中文/数学/代码）巨大。所以现在没人愿意用 5 年前的 tokenizer。
4. **写 prompt 的实战 tips**：
   - 中文项目优先用 GPT-4o (o200k) / Qwen / DeepSeek，不要用 GPT-3.5 (cl100k)
   - 数字密集场景看 tokenizer 是否「数字分组化」
   - 大段重复内容（代码 boilerplate）启用 prompt caching
5. **生僻字怎么处理？**
   - BPE 退化到字节级，一个生僻字往往拆成 3 个字节 token。模型还能处理但「贵」+「认不准」。

改一改：用一段你自己最常发的对话作为输入，看你日常实际 token 成本是多少。

## 自检 ✅

- [ ] 解释 BPE 的核心步骤（字节分 → 迭代合并）。
- [ ] 不查文档说出「中文用 GPT-2 比 GPT-4o 大概贵几倍」（≈ 2~3 倍）。
- [ ] 解释「空格归属」对写 prompt 的影响。
- [ ] 解释「为什么 vocab 不是越大越好」（embedding 表 / softmax 成本）。
- [ ] 给一段任意中文，能现场跑出三家 token 数对比。

## 下一步

→ [`12_bf16_training.ipynb`](12_bf16_training.ipynb)